# SmartHVAC Studio: Backend Worker (Layer 3)
This executable notebook acts as the **Coordination Engine**. It polls Firebase for new jobs, runs AI generation (Layer 4), executes EnergyPlus simulations (Layer 5), and uploads results.

---

## 1. Setup Environment

In [ ]:
# 1. Install Backend Dependencies
!pip install -r requirements.txt

# 2. Install Advisor's EnergyPlus Utility (from GitHub)
# Note: This installs the specific dev branch required for EKF hooks
!pip install -q "energy-plus-utility @ git+https://github.com/mugalan/energy-plus-utility.git@dev"

print("Dependencies installed.")

In [ ]:
import sys
import os

# Add the local folder to path so we can import 'backend'
sys.path.append(os.getcwd())

print("Current Working Directory:", os.getcwd())
print("Python Path Updated.")

## 2. Authentication

In [ ]:
# Check for Service Account Key
key_path = "serviceAccountKey.json"

if not os.path.exists(key_path):
    print("❌ ERROR: serviceAccountKey.json not found.")
    print("Please upload your Firebase Service Account JSON key to the file browser on the left.")
else:
    print("✅ Found serviceAccountKey.json")

## 3. Initialize Modules

In [ ]:
from backend.firebase_connector import FirebaseConnector
from backend.ai_generator import AIPipelines
from simulation_runner import run_simulation_job
import time

# Initialize Connections
try:
    fb = FirebaseConnector(key_path)
    ai = AIPipelines() # API Key can be passed here or set in ENV
    print("Backend Modules Initialized Successfully.")
except Exception as e:
    print("Initialization Failed:", e)

In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore, storage
import json
import time
import difflib # For Diff Viewer

## 4. Verify epw found and downloaded to runtime

In [ ]:
import os

# Expected path relative to Colab root
file_path = "weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw"

if os.path.exists(file_path):
    print(f"✅ FOUND: {file_path}")
    print(f"   Size: {os.path.getsize(file_path)/1024:.1f} KB")
else:
    print(f"❌ MISSING: {file_path}")
    if os.path.exists("weather"):
        print(f"   Contents of 'weather/': {os.listdir('weather')}")
    else:
        print("   'weather' folder does not exist!")


## 5. Verify Base.idf found and downloaded to runtime

In [ ]:
import os
import firebase_admin
from firebase_admin import storage

def download_template(template_name="Base.idf"):
    """Downloads the IDF template from Firebase Storage to local Colab."""
    try:
        bucket = storage.bucket()
        blob_path = f"templates/{template_name}"
        local_path = f"templates/{template_name}"

        # Ensure local dir exists
        os.makedirs("templates", exist_ok=True)

        blob = bucket.blob(blob_path)
        if blob.exists():
            blob.download_to_filename(local_path)
            print(f"✅ Downloaded Template: {local_path}")
            return True
        else:
            print(f"⚠️ Template not found in Storage: {blob_path}")
            print("   (Please create 'templates' folder in Firebase Storage and upload Base.idf)")
            return False

    except Exception as e:
        print(f"❌ Failed to download template: {e}")
        return False

        return False

def generate_and_upload_diff(base_path, new_path, job_id):
    """Generates an HTML diff between Base.idf and in.idf and uploads it."""
    try:
        # Read files
        with open(base_path, 'r') as f:
            base_lines = f.readlines()
        with open(new_path, 'r') as f:
            new_lines = f.readlines()

        # Generate HTML Diff using Python's built-in difflib
        diff_html = difflib.HtmlDiff().make_file(
            base_lines, new_lines,
            fromdesc='Base Template',
            todesc='Generated IDF',
            context=True,  # Show only changes + context
            numlines=3     # 3 lines of context
        )

        # Save locally
        diff_path = f"jobs/{job_id}/diff.html"
        with open(diff_path, "w") as f:
            f.write(diff_html)

        # Upload to Storage (Content-Type is CRITICAL for browser viewing)
        bucket = storage.bucket()
        blob = bucket.blob(f"jobs/{job_id}/diff.html")
        blob.upload_from_filename(diff_path, content_type="text/html")
        print(f"   📊 Uploaded Diff HTML to Storage.")
        return True
    except Exception as e:
        print(f"   ❌ Failed to generate diff: {e}")
        return False

# Call this on startup
download_template()


## 6. Main polling loop

In [ ]:
import time
from datetime import datetime
import traceback

def run_backend_loop(db):
    print("🚀 Backend is running... Waiting for jobs.")
    print("   - Watching 'jobs' collection for 'queued' tasks.")
    print("   - Watching 'test_connectivity' collection for 'test_connection' tasks.")

    while True:
        try:
            # ---------------------------------------------------------
            # 1. Check for CONNECTION TESTS (Fast Path)
            # ---------------------------------------------------------
            test_docs = db.collection("test_connectivity").where("status", "==", "test_connection").stream()

            for doc in test_docs:
                job_id = doc.id
                data = doc.to_dict()
                print(f"🔹 Found Connection Test: {job_id}")

                # Get Toggles (Default to True if missing)
                do_gemini = data.get('checkGemini', True)
                do_openai = data.get('checkOpenAI', True)
                do_hf = data.get('checkHF', True)

                # Perform checks using your existing AI class
                # Pass the flags to the method
                results = ai.test_connections(check_openai=do_openai, check_gemini=do_gemini, check_hf=do_hf)

                # Update Firestore
                db.collection("test_connectivity").document(job_id).update({
                    "status": "tested",
                    "testResults": results,
                    "processedAt": datetime.now()
                })
                print(f"   ✅ Test Completed. Updated {job_id}")

            # ---------------------------------------------------------
            # 2. Check for SIMULATION JOBS (Normal Path)
            # ---------------------------------------------------------
            job_docs = db.collection("jobs").where("status", "==", "queued").stream()

            for doc in job_docs:
                job_id = doc.id
                data = doc.to_dict()
                print(f"🔸 Found Queued Job: {job_id}")

                # Mark as running immediately
                db.collection("jobs").document(job_id).update({"status": "running"})

                try:
                    # Extract Inputs from new UI
                    nlp_input = data.get("nlpInputText", "")
                    sim_settings = data.get("simulationConfig", {})

                    # Log what we received
                    print(f"   Input: {nlp_input[:50]}...")
                    print(f"   Settings: {sim_settings}")

                    # --- CALL YOUR EXISTING LOGIC HERE ---
                    # result_path = run_experiment(nlp_input, sim_settings)

                    # Mock Success for now (Replace with real logic)
                    time.sleep(2) # Simulate work

                    # Update Job
                    db.collection("jobs").document(job_id).update({
                        "status": "done",
                        "resultPath": "results/mock_path", # Placeholder
                        "completedAt": datetime.now()
                    })
                    print(f"   ✅ Job Finished: {job_id}")

                except Exception as e:
                    error_msg = str(e)
                    print(f"   ❌ Job Failed: {error_msg}")
                    traceback.print_exc()

                    db.collection("jobs").document(job_id).update({
                        "status": "error",
                        "errorMessage": error_msg
                    })

            # Sleep to prevent quota exhaustion
            time.sleep(3)

        except Exception as main_e:
            print(f"⚠️ Loop Error: {main_e}")
            time.sleep(5)

# ---------------------------------------------------------------------
# START THE LOOP (Runs indefinitely)
# Ensure 'fb' is defined (from previous cells)
# ---------------------------------------------------------------------
if __name__ == "__main__":
    if 'fb' in globals():
        # Access the internal firestore client from your connector
        if hasattr(fb, 'db'):
            db_client = fb.db
        else:
            # Fallback if fb is the client itself
            db_client = fb

        run_backend_loop(db_client)
    else:
        print("❌ Error: 'fb' variable not found. Please run the Firebase Initialization cell first.")
